# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [33]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [34]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['BIZVDJJMWY', 'LRKPDDWHBI'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 2,  9, 26, 22,  4, 10, 10, 13, 23, 25],
       [12, 18, 11, 16,  4,  4, 23,  8,  2,  9]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 25, 23, 13, 10, 10,  4, 22, 26,  9],
       [ 0,  9,  2,  8, 23,  4,  4, 16, 11, 18]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[25, 23, 13, 10, 10,  4, 22, 26,  9,  2],
       [ 9,  2,  8, 23,  4,  4, 16, 11, 18, 12]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [35]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        '''
        计算注意力权重的全连接层
        score = h_dec · W · h_enc^T
        h_dec：解码器当前隐藏状态，形状 (batch, hidden)
        h_enc：编码器所有时间步输出，形状 (batch, enc_len, hidden)        
        W：可学习的权重矩阵（由 dense_attn 实现）
        '''
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        todo
        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # 编码器
        enc_emb=self.embed_layer(enc_ids)                   # (batch, enc_len, 64)
        enc_outputs,enc_state=self.encoder(enc_emb)         # enc_outputs: (batch, enc_len, 128)
                                                            # enc_state: (batch, 128)
        # 解码器初始化
        dec_emb=self.embed_layer(dec_ids)                   # (batch, dec_len, 64)
        batch_size=tf.shape(dec_emb)[0]
        dec_len=tf.shape(dec_emb)[1]

        decoder_state=enc_state # 初始化解码器状态为编码器最终状态

        outputs_ta=tf.TensorArray(dtype=tf.float32,size=dec_len, clear_after_read=False)
        # all_outputs=[] # 存储每个时间步的输出

        # 逐时间步解码（带注意力）
        for t in tf.range(dec_len):
            current_input=dec_emb[:,t,:] # 当前时间步的输入 (batch, 64)
            decoder_output,decoder_state=self.decoder_cell(current_input,decoder_state) # decoder_output: (batch, 128)
            # 双线性注意力
            query=self.dense_attn(decoder_output) # 将 decoder_output 投影到与 enc_outputs 相同的空间，(batch, 128)
            query=tf.expand_dims(query,axis=1)              # (batch, 1, 128)
            # 计算注意力分数
            scores=tf.matmul(query,enc_outputs,transpose_b=True)  # (batch, 1, enc_len)
            scores=tf.squeeze(scores,axis=1)                      # (batch, enc_len)
            # 归一化得到注意力权重
            attn_weights=tf.nn.softmax(scores,axis=-1)            # (batch, enc_len)
            attn_weights=tf.expand_dims(attn_weights,axis=-1)     # (batch, 1, enc_len)

            # 计算上下文向量
            context=tf.einsum('bni,bnh->bh',attn_weights,enc_outputs)
            # context=tf.matmul(attn_weights,enc_outputs)           # 计算上下文向量（加权和）
            # context=tf.squeeze(context,axis=1)                    # (batch, 128)

            # 将解码器输出与上下文向量拼接
            combined=tf.concat([decoder_output,context],axis=-1)  # (batch, 256)
            logits=self.dense(combined)                           # (batch, 27)

            outputs_ta=outputs_ta.write(t,tf.expand_dims(logits,axis=1))
            # all_outputs.append(tf.expand_dims(logits,axis=-1))    # 保持维度用于序列输出

        # logits=tf.concat(all_outputs,axis=1)                      # (batch, dec_len, 27)
        stacked=outputs_ta.stack()
        logits=tf.squeeze(stacked,axis=2)
        logits=tf.transpose(logits,[1,0,2])           # 转换为 (batch, dec_len, vocab_size)
        
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, enc_state
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
        # 1. 嵌入当前输入
        inp_emb=self.embed_layer(x)                                # (batch, 64)
        # 2. RNN 单步计算
        decoder_output,new_state=self.decoder_cell(inp_emb,state)  # (batch, 128)
        # 3. 计算注意力
        query=self.dense_attn(decoder_output)
        query=tf.expand_dims(query,axis=1)

        scores=tf.matmul(query,enc_out,transpose_b=True)
        scores=tf.squeeze(scores,axis=1)

        attn_weights=tf.nn.softmax(scores,axis=-1)
        attn_weights=tf.expand_dims(attn_weights,axis=1)

        context=tf.matmul(attn_weights,enc_out)
        context=tf.squeeze(context,axis=1)

        combined=tf.concat([decoder_output,context],axis=-1)

        logits=self.dense(combined)

        out=tf.argmax(logits,axis=-1)
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''
        return out, new_state

# Loss函数以及训练逻辑

In [36]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [37]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.2996922
step 500 : loss 1.6738304
step 1000 : loss 0.28660572
step 1500 : loss 0.029077992


<tf.Tensor: shape=(), dtype=float32, numpy=0.017078009>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [38]:
def sequence_reversal():
    def decode(enc_state, steps, enc_out):
        b_sz = tf.shape(enc_state)[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = enc_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, enc_state = model.encode(enc_x)
    predictions = decode(enc_state, enc_x.shape[-1], enc_out)
    return predictions, batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False]
[('LRJXRDHYUOWYEPYSCHGK', 'KGHCSYPEYWOUYHDRXJRL'), ('KOWEVLSPYHEFEVSFPFQZ', 'ZQFPFSVEFEHYPSLVEWOK'), ('AYTLAUICBCWRRXOYVFGY', 'YGFVYOXRRWCBCIUALTYA'), ('SMPTKWCSJMETAVXDBHWI', 'IWHBDXVATEMJSCWKTPMS'), ('IEODTOPQWPPJLSDVCOOI', 'IOOCVDSLJPPWQPOTDOEI'), ('KYRPLXPHCFKSMFAOCNQV', 'VQNCOAFMSKFCHPXLPRYK'), ('DRYSRAYJBWCYCAUDGVMR', 'RMVGDUACYCWBJYARSYRD'), ('VWLEXBMSRVPVNZVSWVZC', 'CZVWSVZNVPVRSMBXELWV'), ('HKKITMYEAKPSDEICJUUT', 'TUUJCIEDSPKAEYMTIKKH'), ('MKTQIBMONADTYBNPHVHJ', 'JHVHPNBYTDANOMBIQTKM'), ('OMLVWEGLBLTAECHIYFNB', 'BNFYIHCEATLBLGEWVLMO'), ('UVNEKGETJZQCCGJFRZTT', 'TTZRFJGCCQZJTEGKENVU'), ('GDBKKYYBKSOLHJFOGJDG', 'GDJGOFJHLOSKBYYKKBDG'), ('EWUBLVEPLKLTOROKNJOH', 'HOJNKOROTLKLPEVLBUWE'), ('UDIVVQAODZBFSAEMUBDC', 'CDBUMEASFBZDOAQVVIDU'), ('XXKANCPCTHVHIKAMATWN', 'NWTAMAKIHVHTCPCNAKXX'), ('TQF